# Chapter 2 - Lab 3: Portfolio Rebalancing Agent with Episodic Memory

## Notebook Overview

This notebook is the **episodic memory** follow-up to Lab 2's working-memory lab, and it reuses Lab 2's exact setup: the same portfolio-rebalancing agent and the same six-turn conversation about shifting allocations in different market regimes. Where Lab 2 stopped at showing that memory survives *within* a session, this lab goes the whole way and shows that memory can survive *across* sessions. The conversation is condensed into a single structured "episode," that episode is embedded and stored in a vector database, and then the notebook proves the memory works by opening a brand-new conversation — with no history of its own — in which the agent still recalls what happened before. If you only run one of Labs 2 and 3, this is the one to run.

After the shared installs and the `.env` key loading, the first half of the notebook is deliberately familiar: the same bare `portfolio_agent` (`gpt-4o-mini`, no tools) that interprets market volatility and trend into explainable rebalancing recommendations, the same `SQLiteSession("conversation_ptf")` acting as its working memory, and the same queries walking the allocation from 70/20/10 down to 50/35/15, 45/40/15, and back up through 55/30/15 to 70/25/05. The working-memory display cells after each turn show the stored history accumulating, and a short section explains why working memory matters. This shared preamble is what makes the episodic half meaningful: the episode is produced from exactly this conversation.

The heart of the lab, under **Generate Episodic Memory**, begins with a hand-written prompt template. The template addresses an "episodic memory agent" and asks it to read a conversation and return, as strict JSON, five fields: `conversation_summary`, `portfolio_allocation_trajectory`, `what_worked`, `what_didnt_work` and `key_points`. It includes a good and a bad few-shot example so the model knows both the desired structure and the mistakes to avoid. The conversation is then pulled out of the session with `get_items()`, flattened into a single `conversation_history_string` of alternating role and content blocks, and fed to the template.

Two methods are shown for turning that prompt into an episode, so you can compare them side by side. **Method 1 — LangChain:** the template is wrapped in `ChatPromptTemplate.from_template(...)`, combined with `ChatOpenAI(temperature=0.7, model="gpt-4o-mini")` and a `JsonOutputParser()`, then invoked with the flattened conversation to return the episode directly as a Python dict. **Method 2 — the OpenAI Responses API:** the same template text is formatted into a user prompt and sent to `client.responses.create(...)` with JSON output requested; the text of the first response item is then parsed with `json.loads`. Both routes produce the same structured episode, and the responses-API cell is explicitly kept separate (a dedicated `import json` belongs before it if you run the notebook top to bottom).

Storage comes next under **Storing the Episodic Memory**. The notebook installs ChromaDB and opens a `PersistentClient(path="./chroma_ptf_rebal")`, which writes a folder right next to the notebook, so the episodes live on disk rather than in memory. An OpenAI embedding function using `text-embedding-3-small` is attached, and a collection named `episodic_memory_ptf_rebal` is created; a `get_model()` cell prints the embedding parameters for inspection. The generated episode is then composed into a single `document_text` (conversation summary, trajectory, what worked, what didn't, key points) and added to the collection by ID, meaning the episode is stored twice in complementary ways — as readable strings in the document, and as a dense embedding vector alongside it.

The **Query Memory** section demonstrates recall. A new user query about a low-volatility, bullish market is embedded and matched against the collection with `collection.query(query_texts=[...], n_results=3)`, returning the most relevant past episodes. The **Call the agent** section then puts that retrieval to work: it creates a brand-new session (`conversation_ptf_2`) with zero history, builds a prompt that says, in effect, "here are relevant insights from past rebalancing episodes," appends the retrieved recap to the new query, and runs the agent. The agent responds using the last allocation from the stored episode — information that exists only in the vector database, not in the new session. That single answer is the entire demonstration: episodic memory has carried knowledge across two conversations that share nothing but the persistent store.

A closing **All in** section packages everything into four reusable functions — `concat_working_memory`, `get_episodic_memory`, `add_data_to_episodic_memory` and `get_similar_episodes` — and runs an interactive loop under a third session: each `User:` prompt is augmented with its most similar episodes before reaching the agent, an empty input is gracefully skipped, and typing `exit` triggers the summarization and the ChromaDB write under collection id `memory_2`. Inspection cells then show the stored document and its embeddings. Two final sections, **Semantic Memory** and **Procedural Memory**, are conceptual only: they map the same retrieve-then-append pattern onto factual knowledge bases and onto "how-to" procedural workflows.

**The point of the lab:** working memory dies with its session; episodic memory does not. By summarizing, embedding and storing a conversation, the agent can *learn from past sessions* and apply those lessons to conversations it has never seen — which is exactly the long-term-memory capability an autonomous agent needs to behave consistently over time.


In this tutorial, we'll use an agent responsible for generating clear and explainable portfolio rebalancing recommendations.

First, we will build a discussion with the agent: **Conversation History**. This will constitute our **working memory**. This section is based on the implementation introduced in Chapter 2 – Lab 2. We include it here to construct our episodic memory.

Next, we will use a specific prompt to extract the most important information from this conversation history to build our **Episodic Memory**:
* A summary of the conversation
* A trajectory of the ptf rebalancing allocations depending on the various market situations
* What worked
* What didn't work
* Key insights

We will store then all this information in a vector database ==> **ChromaDB**

In the following conversations, we will extract from the vector DB the information that is most relevant to the user's query.

The retrieved items correspond to past episodes that are inserted into the prompt before issuing the new request to the agent. This allows the agent to evaluate the query not only based on its own capabilities, but also by leveraging relevant past experiences.


**Key components of long-term memory:**

* Episodic Memory = Specific experiences from the past

* Semantic memory = Factual knowledge

* Procedural memory = "How to" for example "How to perform a KYC check"

# Install libs

In [1]:
# !pip install openai-agents -q

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [3]:
from agents import Agent, Runner, function_tool, SQLiteSession
import nest_asyncio
nest_asyncio.apply()

# Rebalancing Portfolio Agent

Create portfolio rebalancing agent.

In this agent, I didn't add tools to interact with or any other reflective mechanisms. It's a simple agent that I'll be using to build a memory through the conversation.


In [4]:
portfolio_agent = Agent(
    name="Portfolio Rebalancing Agent",
    instructions="""You are a Portfolio Rebalancing Agent for a professional investor.
Your goal is to propose clear, explainable rebalancing actions that keep the portfolio aligned with the target risk/return profile and constraints.

Your role
- Analyze the current portfolio (weights, asset classes, sectors, factors, cash).
- Incorporate market regime information (volatility, trends, macro environment).
- Compare current allocation to:
    - Target allocation or risk budget (if provided),
    - Or to a reasonable diversified allocation based on the information you have.
- Propose specific rebalancing trades with approximate percentage changes and short rationales.
- Prioritize diversification, risk control, and alignment with the investor’s objectives.
- Keep answers concise and structured, and state assumptions when information is missing.
- Treat the allocation provided in my prompot as the portfolio's current allocation.

Always explain your reasoning.""",
    model="gpt-4o-mini"
)


You can of course imagine an agent using tools connecting to:
* Your database (or other sources), to extract the current situation of your portfolio.
* Market data to compute volatility, trend....

This way, your agent can analyze your current portfolio allocation and recommend appropriate rebalancing actions using real market data.

# Set up Working Memory

Let’s instantiate the working memory, referred to as a session in the OpenAI Agent SDK:

In [5]:
# Create a session instance that will persist across runs
session_id = "conversation_ptf"
session_ptf_rebal = SQLiteSession(session_id)

The SQLiteSession class provides a session mechanism that stores the interaction history in a SQLite database, enabling persistence between runs. The variable session_id uniquely names the session and allows the system to associate stored messages and state with this specific conversation.

## User Query 1

We'll add the session memory parameter to the agent's run. This ensures that the entire interaction is stored and maintained as part of the working memory.

We provide the agent with the current market context, such as the volatility regime and market trend, along with the current portfolio allocation, and ask it to recommend appropriate rebalancing actions.


In [6]:
input_user_1 = """Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?"""

output_1 = await Runner.run(
    starting_agent=portfolio_agent,
    input=input_user_1,
    session=session_ptf_rebal
)

print(f"Agent Decision:\n{output_1.final_output}\n")

Agent Decision:
### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potential losses.

2. **Increase Bond Exposure:**
   - **Action:** Increase bonds from 20% to 30%.
   - **Rebalance:** Buy 10% in bonds from proceeds of stock sales.
   - **Rationale:** Bonds can provide stability and cash flow during market downturns.

3. **Increase Cash Holdings:**
   - **Action:** Increase cash from 10

The agent begins by outlining the current portfolio allocation, then states its assumptions and presents a recommended target allocation by asset class. It also provides a detailed rationale for each proposed adjustment:
* Reduce stock allocation from 70% to 50%
* Increase bond allocation from 20% to 40%
* Maintain cash at 10%


### Display Working Memory

Here we display the working memory at this point in the interaction with the agent:

In [7]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potentia

## User Query 2

We now ask a follow-up question while keeping the session parameter during execution, ensuring that the conversation history is preserved:

In [8]:
input_user_2="In this case, I'm reducing equity to 50%, increasing bonds to 35% and cash to 15%."

output_2 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_2,
        session=session_ptf_rebal
    )

print(f"{output_2.final_output}\n")

### Updated Proposed Target Allocation:
Considering your revised target allocation:
- **Stocks:** 50%
- **Bonds:** 35%
- **Cash:** 15%

### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** Mitigating risk in a declining market to better align with your target allocation.

2. **Increase Bond Exposure:**
   - **Action:** Increase bonds from 20% to 35%.
   - **Rebalance:** Buy 15% in bonds from proceeds of stock sales.
   - **Rationale:** Enhancing bond allocation offers more stability and returns, counteracting stock volatility.

3. **Increase Cash Holdings:**
   - **Action:** Increase cash from 10% to 15%.
   - **Rebalance:** Allocate an additional 5% from stock sales to cash.
   - **Rationale:** Higher cash levels improve liquidity and provide a buffer against cont

### Display Working Memory

We now display the  agent’s working memory following the execution of the  two previous queries:

In [9]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potentia

As you can see, all the discussion is stored in the working memory (session).

## User Query 3

Here is another request, still with the session parameter specified during the run:

In [10]:
input_user_3 = """Current Market Situation:
- Volatility Regime: high
- Market Trend: uncertain

What rebalancing action should I take?"""

output_3 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_3,
        session=session_ptf_rebal
    )

print(f"{output_3.final_output}\n")

### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 50%
  - Bonds: 35%
  - Cash: 15%

### Market Situation Overview:
- **Volatility Regime:** High
- **Market Trend:** Uncertain

In an uncertain market combined with high volatility, it is prudent to maintain a defensive stance while allowing for some growth potential. 

### Suggested Target Allocation:
Given the need for balance:
- **Stocks:** 45%
- **Bonds:** 40%
- **Cash:** 15%

### Proposed Rebalancing Actions:
1. **Slightly Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 50% to 45%.
   - **Rebalance:** Sell 5% of stocks.
   - **Rationale:** Reducing equity exposure helps to manage risk while still allowing for growth potential in a volatile market.

2. **Increase Bond Exposure:**
   - **Action:** Increase bonds from 35% to 40%.
   - **Rebalance:** Buy 5% in bonds with the proceeds from stock sales.
   - **Rationale:** Higher bond allocation provides stability and income during periods of uncertainty.

### Display Working Memory

In [11]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potentia

## User Query 4

In [12]:
input_user_4 = """In this case, I reduce equity to 45%, increase bonds to 40% and keep cash unchanged"""

output_4 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_4,
        session=session_ptf_rebal
    )

print(f"{output_4.final_output}\n")

### Updated Proposed Target Allocation:
- **Stocks:** 45%
- **Bonds:** 40%
- **Cash:** 15%

### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 50%
  - Bonds: 35%
  - Cash: 15%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 50% to 45%.
   - **Rebalance:** Sell 5% of stocks.
   - **Rationale:** This reduction aligns with your target allocation while controlling exposure to market risk during uncertain times.

2. **Increase Bond Exposure:**
   - **Action:** Increase bonds from 35% to 40%.
   - **Rebalance:** Use the proceeds from the stocks sold to buy 5% more in bonds.
   - **Rationale:** An increased allocation to bonds enhances income and stability, crucial in high-volatility environments.

3. **Cash Remains Steady:**
   - **Action:** Keep cash at 15%.
   - **Rationale:** Maintaining cash levels provides liquidity and a buffer against market fluctuations.

### Summary of Transactions:
- **Sell Stocks:** 5% (to a

### Display Working Memory

In [13]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potentia

## User Query 5

In [14]:
input_user_5 = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

output_5 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_5,
        session=session_ptf_rebal
    )

print(f"{output_5.final_output}\n")

### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 45%
  - Bonds: 40%
  - Cash: 15%

### Market Situation Overview:
- **Volatility Regime:** Low
- **Market Trend:** Bullish

In a low-volatility environment with a bullish market trend, it’s advantageous to increase equity exposure to capitalize on potential growth opportunities.

### Suggested Target Allocation:
Given the positive market conditions, consider adjusting your allocation as follows:
- **Stocks:** 60%
- **Bonds:** 30%
- **Cash:** 10%

### Proposed Rebalancing Actions:
1. **Increase Stock Exposure:**
   - **Action:** Increase stocks from 45% to 60%.
   - **Rebalance:** Buy 15% in stocks.
   - **Rationale:** This aligns with the bullish trend and allows you to capture growth potential while taking advantage of low volatility.

2. **Reduce Bond Exposure:**
   - **Action:** Decrease bonds from 40% to 30%.
   - **Rebalance:** Sell 10% of bonds.
   - **Rationale:** With a low-volatility and bullish environment, 

## User Query 6

In [15]:
input_user_6 = """in this situation, I'll increase stock to 70%, reduce bonds to 25% and cash to 5%"""

output_6 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_6,
        session=session_ptf_rebal
    )

print(f"{output_6.final_output}\n")

### Updated Proposed Target Allocation:
- **Stocks:** 70%
- **Bonds:** 25%
- **Cash:** 5%

### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 45%
  - Bonds: 40%
  - Cash: 15%

### Proposed Rebalancing Actions:
1. **Increase Stock Exposure:**
   - **Action:** Increase stocks from 45% to 70%.
   - **Rebalance:** Buy 25% in stocks.
   - **Rationale:** With a bullish market trend and low volatility, increasing exposure to stocks is appropriate to maximize growth potential.

2. **Reduce Bond Exposure:**
   - **Action:** Decrease bonds from 40% to 25%.
   - **Rebalance:** Sell 15% of bonds.
   - **Rationale:** Lowering bond allocation allows capital to be redirected to equities where growth potential is higher in the current market.

3. **Decrease Cash Holdings:**
   - **Action:** Reduce cash from 15% to 5%.
   - **Rebalance:** Sell 10% of cash to fund additional stock purchases.
   - **Rationale:** In a strong market, maintaining minimal cash is effective for capturing gr

### Display Final Working Memory

In [16]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potentia

In [17]:
# conversation_history_string = "/n".join([f"{elem[0]}: {elem[1]}" for elem in conversation_history])

# conversation_history_string+='user: p'

You can play with the agent yourself:

In [18]:
# input_user_7 = """ Write here your prompt
# """

# output_7 = await Runner.run(
#         starting_agent=portfolio_agent,
#         input=input_user_7,
#         session=session_ptf_rebal
#     )

# print(f"{output_7.final_output}\n")

Working memory is essential in this context.

Without it, the agent would not retain the initial portfolio allocation, {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}, and would therefore be unable to correctly apply subsequent instructions, such as increasing bonds to 25%, and reducing cash to 5%.

Without access to the initial state, the agent would not compute or recommend an appropriate rebalancing strategy.

# Generate Episodic Memory

I'll show here two methods to generate the episodic memory:

* Using LangChain API
* Using Responses API from OpenAI

## Prompt for Episodic Memory

 We design a detailed prompt that provides clear instructions on the information to extract. We also include examples using a few-shot approach, illustrating what constitutes good and poor summaries, as well as effective and ineffective outcomes.


In [19]:
episodic_memory_prompt_template = """You are an episodic memory agent supporting a principal portfolio-rebalancing agent.
Your job is to create a concise episodic memory of the session:
* Summarize the conversation (in conversation_summary),
* Store a structured trajectory of portfolio allocations and their corresponding market situations, capturing how allocations evolved over time and under which conditions (in portfolio_trajectory).
* Identify what worked (in what_worked) and what didn’t (in what_didnt_work),
* Extract key points (in key_points).

You receive the full conversation history between the user and the principal agent.
The portfolio includes equities, bonds, and cash, and the discussion covers the market situation, portfolio state and outcome, and rebalancing actions.

OUPUT FORMAT:
Return only valid JSON in this exact schema:
{{

	"conversation_summary": string, // 3–5 sentences summarizing: market conditions, portfolio situation, actions taken or discussed.
	"portfolio_allocation_trajectory": string, // Portfolio trajectory recording how portfolio allocations evolve across sessions together with the associated market situations.
	"what_worked": string, //Short analysis of what was effective: alignment with goals, good risk management, coherent rebalancing reasoning, improved diversification, clear decisions.
	"what_didnt_work": string, //Short analysis of frictions: unclear strategy, missed opportunities, over/under-reaction to market signals, concentration risk, unclear constraints, data issues
	"key_points": ["string"] // List of the most important insights or decisions to keep for future sessions.
}}


FEW-SHOT EXAMPLES
Here are some examples of what is good to produce and what is bad to avoid:

GOOD conversation_summary — Example
“Market uncertainty increased due to rising yields and mixed earnings. The portfolio was equity-heavy at 65%, creating higher volatility. The user discussed reducing equity risk and reallocating toward short-duration bonds. A partial rebalance was executed, bringing equities to 55% and increasing cash reserves. The agent emphasized maintaining flexibility given macro uncertainty.”

BAD conversation_summary — Example
“The market was bad. The portfolio changed. Some trades were done. It was about rebalancing.”
(Too vague, no numbers, no context, no actions.)

GOOD what_worked — Example
“The shift from long-duration bonds to shorter maturities improved interest-rate resilience. The reduction of tech overweight aligned well with the user’s lower risk tolerance. The conversation clearly linked macro signals to rebalancing decisions.”

BAD what_worked — Example
“Everything worked well.” (Empty, no link to portfolio or market.)
“The agent was nice.” (Irrelevant.)

GOOD what_didnt_work — Example:
“The user’s liquidity needs were mentioned but not fully integrated into the new allocation. Equity risk remains high despite concerns about earnings volatility. Market data gaps created hesitation about bond exposure.”

BAD what_didnt_work — Example:
“Nothing went wrong.” (Unrealistic; lacks analysis.)
“The user disagreed with the agent.” (Not informative, no portfolio implications.)

General Rules:
- Be concise and factual.
- Do not invent trades or market context not present in the conversation.
- Capture insights, not generic statements.
- Output only valid JSON and nothing else.

Here is the conversation between the user and the principal portfolio-rebalancing agent:
{conversation}
"""

## Working Memory Recap: Conversation History

In [20]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?
----------------------------------------------------------------------------------------------------
  2. assistant: ### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potentia

### Concat conversation history

In [21]:
conversation_history_string = "\n\n".join([f'{elem[0]}: """{elem[1]}"""' for elem in conversation_history])

print(conversation_history_string)

user: """Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?"""

assistant: """### Current Portfolio Analysis:
- **Current Allocation:**
  - Stocks: 70%
  - Bonds: 20%
  - Cash: 10%

- **Target Characteristics:**
  - Given high volatility and a declining market, a defensive positioning is warranted. 

### Suggested Target Allocation:
To align with the current market situation, consider a more conservative target allocation:
- **Stocks:** 50%
- **Bonds:** 30%
- **Cash:** 20%

### Proposed Rebalancing Actions:
1. **Reduce Stock Exposure:**
   - **Action:** Decrease stocks from 70% to 50%.
   - **Rebalance:** Sell 20% of stocks.
   - **Rationale:** In a high-volatility and declining market, reducing stock exposure mitigates risk and potential losses.

2. **Increase Bond Exposure:**
   - **Action:** Increase bonds from 20% to 30%.
   - **Reb

## Method 1: LangChain

Install packages

In [22]:
!pip install langchain_openai -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openai-agents 0.22.1 requires openai<4,>=3.0.0, but you have openai 2.54.0 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0.7, model="gpt-4o-mini")

### LangChain prompt template

In [25]:
episodic_memory_prompt = ChatPromptTemplate.from_template(episodic_memory_prompt_template)

generate_episodic_memory = episodic_memory_prompt | llm | JsonOutputParser() # Chaining prompt, llm and structured output ==> to be used to generate the episodic memory

In [26]:
print(episodic_memory_prompt.messages[0].prompt.template)

You are an episodic memory agent supporting a principal portfolio-rebalancing agent.
Your job is to create a concise episodic memory of the session:
* Summarize the conversation (in conversation_summary),
* Store a structured trajectory of portfolio allocations and their corresponding market situations, capturing how allocations evolved over time and under which conditions (in portfolio_trajectory).
* Identify what worked (in what_worked) and what didn’t (in what_didnt_work),
* Extract key points (in key_points).

You receive the full conversation history between the user and the principal agent.
The portfolio includes equities, bonds, and cash, and the discussion covers the market situation, portfolio state and outcome, and rebalancing actions.

OUPUT FORMAT:
Return only valid JSON in this exact schema:
{{

	"conversation_summary": string, // 3–5 sentences summarizing: market conditions, portfolio situation, actions taken or discussed.
	"portfolio_allocation_trajectory": string, // Po

### Generate Episodic Memory with LangChain

In [27]:
episodic_memory = generate_episodic_memory.invoke({"conversation": conversation_history_string})
episodic_memory

{'conversation_summary': 'The market has shifted to a low volatility and bullish trend, prompting a reevaluation of the portfolio. The current allocation was 45% stocks, 40% bonds, and 15% cash. The user decided to increase stock exposure to 70%, reduce bonds to 25%, and decrease cash to 5%. This aggressive rebalancing aims to capitalize on growth opportunities in the favorable market conditions.',
 'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% | Session 2: Stocks 50%, Bonds 30%, Cash 20% | Session 3: Stocks 45%, Bonds 40%, Cash 15% | Session 4: Stocks 60%, Bonds 30%, Cash 10% | Session 5: Stocks 70%, Bonds 25%, Cash 5%',
 'what_worked': 'The decision to increase stock exposure in response to a bullish market trend effectively aligns with growth objectives. The reduction in bonds and cash maintains a balanced approach while focusing on maximizing returns during favorable conditions.',
 'what_didnt_work': 'The significant reduction of cash could lead to 

The result follows the structured format specified in the prompt. A summary of the conversation is then generated, along with additional analysis highlighting what worked during execution, what did not, and the key elements of the exchange between the agent and the user.


In [28]:
# You can also add the whole conversation history to the episodic memory.
# Or add only the last 4 messages (user & assistant)
# episodic_memory
# str_json = f"""{'conversation_history': {conversation_history_string},
#   'conversation_summary': 'The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.',
#  'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)',
#  'what_worked': 'The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging potential for higher returns. The strategy demonstrates effective risk management by reducing bonds and cash in favor of equities, optimizing the portfolio for growth.',
#  'what_didnt_work': 'The substantial reduction in bonds and cash may expose the portfolio to greater risk, particularly if market conditions fluctuate unexpectedly. A lower cash reserve could limit liquidity for future opportunities or risk mitigation.',
#  'key_points': ['Leverage bullish market conditions by increasing equity exposure.',
#   'Maintain a balance between risk and potential returns when adjusting allocations.',
#   'Monitor market conditions closely after significant rebalancing actions.']}"""

In [29]:
# episodic_memory is already a dict because the chain ends with JsonOutputParser()
if isinstance(episodic_memory, str):
    str_json_fixed = episodic_memory.replace("'", '"')
    episodic_memory_json = json.loads(str_json_fixed)
else:
    episodic_memory_json = episodic_memory


## Method 2: Responses API from OpenAI

In [ ]:
user_prompt = episodic_memory_prompt_template.format(
    conversation=conversation_history_string
)
print(user_prompt)

You are an episodic memory agent supporting a principal portfolio-rebalancing agent.
Your job is to create a concise episodic memory of the session:
* Summarize the conversation (in conversation_summary),
* Store a structured trajectory of portfolio allocations and their corresponding market situations, capturing how allocations evolved over time and under which conditions (in portfolio_trajectory).
* Identify what worked (in what_worked) and what didn’t (in what_didnt_work), and
* Extract key points (in key_points).

You receive the full conversation history between the user and the principal agent.
The portfolio includes equities, bonds, and cash, and the discussion covers the market situation, portfolio state and outcome, and rebalancing actions.

OUPUT FORMAT:
Return only valid JSON in this exact schema:
{

	"conversation_summary": string, // 3–5 sentences summarizing: market conditions, portfolio situation, actions taken or discussed.
	"portfolio_allocation_trajectory": string, //

In [ ]:
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1-mini",
  input=[{"role":"user","content":user_prompt}],
  text = {"format":{"type": "json_object"}},
)

print(json.dumps(json.loads(response.output[0].content[0].text), indent=2))

{
  "conversation_summary": "The conversation tracked portfolio rebalancing across changing market conditions. Initially, with high volatility and a declining market, the user reduced equity from 70% to 50%, increased bonds and cash, aiming for capital preservation. Later, facing high volatility with uncertain trends, equity was further trimmed to 45%, bonds increased to 40%, and cash remained at 15%, reflecting a cautious but balanced stance. Upon a shift to low volatility and a bullish market, the user decided on a more aggressive portfolio: increasing stocks from 45% to 70%, reducing bonds and cash to capitalize on growth potential while accepting higher risk.",
  "portfolio_allocation_trajectory": "[ { \"market\": { \"volatility\": \"high\", \"trend\": \"declining\" }, \"allocation\": { \"stocks\": 0.70, \"bonds\": 0.20, \"cash\": 0.10 } }, { \"market\": { \"volatility\": \"high\", \"trend\": \"declining\" }, \"allocation\": { \"stocks\": 0.50, \"bonds\": 0.35, \"cash\": 0.15 } }, 

In [ ]:
episodic_memory = json.dumps(json.loads(response.output[0].content[0].text), indent=2)

# Storing the Episodic Memory:

## ChromaDB client

The generated content, "episodic_memory", is embedded using the text-embedding-3-small model from OpenAI and stored in a vector database, ChromaDB, for efficient similarity-based retrieval.


In [ ]:
!pip install chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 3.1 MB/s eta 0

In [ ]:
import chromadb

from chromadb.utils import embedding_functions

# Define an OpenAI embedding function using the specified embedding model
embedding_function  = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY, #Your OpenAI API Key
    model_name="text-embedding-3-small",
)

# Name of the collection used to store episodic memory embeddings
collection_name="episodic_memory_ptf_rebal"

# Create a persistent ChromaDB client with on-disk storage: will be stored in the folder "chroma_ptf_rebal"
persistent_client = chromadb.PersistentClient(path="./chroma_ptf_rebal")

# Create or load the collection and associate it with the embedding function
collection = persistent_client.get_or_create_collection(collection_name, embedding_function=embedding_function )

The collection serves as a container within the vector database where embeddings will be stored.

In [ ]:
#Get the different parameters under the ChromaDB client:
persistent_client.get_collection(name='episodic_memory_ptf_rebal').get_model()

Collection(id=UUID('3956f01b-9d53-40f6-9e80-7afa6e3e8196'), name='episodic_memory_ptf_rebal', configuration_json={'hnsw': {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': {'type': 'known', 'name': 'openai', 'config': {'api_base': None, 'api_key_env_var': 'OPENAI_API_KEY', 'api_type': None, 'api_version': None, 'default_headers': None, 'deployment_id': None, 'dimensions': None, 'model_name': 'text-embedding-3-small', 'organization_id': None}}}, serialized_schema={'defaults': {'string': {'fts_index': {'enabled': False, 'config': {}}, 'string_inverted_index': {'enabled': True, 'config': {}}}, 'float_list': {'vector_index': {'enabled': False, 'config': {'space': 'cosine', 'embedding_function': {'type': 'known', 'name': 'openai', 'config': {'api_base': None, 'api_key_env_var': 'OPENAI_API_KEY', 'api_type': None, 'api_version': None, 'default_headers': None, 'deployment_id': 

## Add Memory in DB

In [ ]:
episodic_memory

{'conversation_summary': 'The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.',
 'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)',
 'what_worked': 'The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging potential for higher returns. The strategy demonstr

In [ ]:
document_text = f"""
Conversation Summary: {episodic_memory['conversation_summary']}
Portfolio Allocation Trajectory: {episodic_memory['portfolio_allocation_trajectory']}
What Worked: {episodic_memory['what_worked']}
What Didn't Work: {episodic_memory['what_didnt_work']}
Key Points: {', '.join(episodic_memory['key_points'])}
"""

metadata_clean = {
    "conversation_summary": episodic_memory["conversation_summary"],
    "portfolio_allocation_trajectory": episodic_memory["portfolio_allocation_trajectory"],
    "what_worked": episodic_memory["what_worked"],
    "what_didnt_work": episodic_memory["what_didnt_work"],
    "key_points": " | ".join(episodic_memory["key_points"]), #Convert a list to a string to be added in chromDB
}

collection.add(
    ids=["memory_1"],  # unique id
    documents=[document_text],  # main text
    metadatas=[metadata_clean],  # full dict as metadata (JSON-serializable)
)


The generated content, `episodic_memeory`, is stored in two complementary ways:
* as formatted text in the documents field
* as structured JSON in the metadatas field.

Together, these two representations capture the full content of the episodic memory while enabling both semantic search (via embeddings) and structured filtering (via metadata).


## Query Memory

Now, consider starting a new session with the agent and submitting a new query.


As a first step, this query is compared against the episodic memory stored in the vector database using similarity-based distance search:


In [ ]:
input_user_init = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

episodic_memory_recap = collection.query(
    query_texts=[input_user_init],
    n_results=3,
)
episodic_memory_recap

{'ids': [['memory_1']],
 'embeddings': None,
 'documents': [["\nConversation Summary: The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.\nPortfolio Allocation Trajectory: Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)\nWhat Worked: The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging p

We query the vector database to retrieve the three most relevant chunks for the user’s query.

Since we have stored only one episodic memory, we retrieve it and incorporate it into the prompt to help the agent answer the user's question.

In [ ]:
# context_memory =
episodic_memory_recap['documents'][0]

["\nConversation Summary: The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.\nPortfolio Allocation Trajectory: Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)\nWhat Worked: The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging potential for higher returns. The strategy demonstrates effec

## Call the agent

As we are starting new session conversation, let's create a new working memory:

In [ ]:
#Create a new session that will persist across runs
session_id = "conversation_ptf_2"
session_ptf_rebal_2 = SQLiteSession(session_id)

In [ ]:
input_user_init = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

input_user_with_memory = input_user_init + f""""

Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:
{episodic_memory_recap['documents'][0][0]}

Argue your proposition using past rebalancing episodes if needed.
"""

output_using_memory = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_with_memory,
        session=session_ptf_rebal_2
    )

print(f"{output_using_memory.final_output}\n")

## Rebalancing Proposal

### Current Portfolio Allocation
- **Stocks**: 70%
- **Bonds**: 25%
- **Cash**: 5%

### Target Risk/Return Profile
- Given the current low volatility and bullish market trend, the focus should continue to be on maximizing equity exposure, but with a cautious approach to maintain some balance and liquidity.

### Suggested Rebalancing Action
- **Adjust Allocations to**:
  - **Stocks**: 65% (decrease by 5%)
  - **Bonds**: 30% (increase by 5%)
  - **Cash**: 5% (no change)
  
### Rationale
1. **Market Conditions**:
   - The current low volatility and bullish environment suggest upside potential for equities. However, given that stocks are already at 70%, it's prudent to dial back slightly to mitigate risk as markets can always shift unexpectedly.

2. **Lessons from Past Episodes**:
   - In previous sessions, maintaining a higher equity allocation worked well during bullish conditions but highlighted the risk of overexposure. For example, during Session 5, the setup 

As you can see, by incoporating the episodic memory, the agent was able to retrieve the last portfolio allocation (from the first interactions with the agent - working memory Part 1): 70% stocks, 25% bonds, 5% cash. Also, the agent was able to retrieve lessons from past episodes and incorporate them in the current answer.

## Display the new working memory

Here we are displaying the new working memory (session_ptf_rebal_2):

In [ ]:
all_items = await session_ptf_rebal_2.get_items()

conversation_history_2 = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history_2.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history_2.append((role, content))
      print("--"*50)

  1. user: Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"

Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:

Conversation Summary: The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.
Portfolio Allocation Trajectory: Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cas

## All in

In this step, we wrap the entire code into functions to improve readability.

* We instantiate a new working session.
* For each input query, we retrieve the most relevant chunks from the episodic memory
* We incorporate these chunks into the prompt, alongside the input query
* The agent answers based on this information
* When we end the discussion with agent, the current working session will be resumed, building the episodic memory, and stored in the vector database, to be used in future interactions.

In [ ]:
def concat_working_memory(all_items):
  conversation_history = []
  for i, msg in enumerate(all_items, 1):
      role = msg.get("role", "unknown")
      content = msg.get("content", "")
      if role == "assistant" :
        conversation_history.append((role, content[0]['text']))
      if role == "user":
        conversation_history.append((role, content))
  return conversation_history

In [ ]:
def get_episodic_memory(conversation_history):
  conversation_history_string = "\n\n".join([f'{elem[0]}: """{elem[1]}"""' for elem in conversation_history])
  episodic_memory = generate_episodic_memory.invoke({"conversation": conversation_history_string})
  return episodic_memory

In [ ]:
def add_data_to_episodic_memory(episodic_memory, collection_id):
  document_text = f"""Conversation Summary: {episodic_memory['conversation_summary']}
  Portfolio Allocation Trajectory: {episodic_memory['portfolio_allocation_trajectory']}
  What Worked: {episodic_memory['what_worked']}
  What Didn't Work: {episodic_memory['what_didnt_work']}
  Key Points: {', '.join(episodic_memory['key_points'])}
  """

  metadata_clean = {
      "conversation_summary": episodic_memory["conversation_summary"],
      "portfolio_allocation_trajectory": episodic_memory["portfolio_allocation_trajectory"],
      "what_worked": episodic_memory["what_worked"],
      "what_didnt_work": episodic_memory["what_didnt_work"],
      "key_points": " | ".join(episodic_memory["key_points"]), #Convert a list to a string to be added in chromDB
  }

  collection.add(
      ids=[collection_id],  # unique id
      documents=[document_text],  # main text
      metadatas=[metadata_clean],  # full dict as metadata (JSON-serializable)
  )

In [ ]:
def get_similar_episodes(input_user_init):
  episodic_memory_recap = collection.query(
    query_texts=[input_user_init],
    n_results=3,
  )

  input_user_with_memory = input_user_init + f"""
    Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:
    {episodic_memory_recap['documents'][0][0]}

    Argue your propsition using past rebalacing episodes if needed.
    """
  return input_user_with_memory

In [ ]:
#Initiate Working Memory: This is another working session
session_id = "conversation_ptf_3"
session_ptf_rebal_3 = SQLiteSession(session_id)

#Define the id collection. Conversation will be stored in the episodic memory when the session ends
collection_id = 'memory_2'
all_items = []

while True:
    # Get User's Input
    input_user_init = input("\nUser: ")

    if input_user_init == "exit":
        if all_items!=[]:
          print("End of the discussion, reformulating working memory to be added to episodic memory")
          conversation_history = concat_working_memory(all_items)
          episodic_memory = get_episodic_memory(conversation_history)
          print("---"*50)
          print("Episodic Memory to be stored: \n")
          print(episodic_memory)
          print("---"*50)
          add_data_to_episodic_memory(episodic_memory, collection_id)
        break

    #Extract episodic memory and build final prompt
    input_user_with_memory = get_similar_episodes(input_user_init)


    output_using_memory = await Runner.run(
            starting_agent=portfolio_agent,
            input=input_user_with_memory,
            session=session_ptf_rebal_3
        )

    print(f"{output_using_memory.final_output}\n")

    all_items = await session_ptf_rebal_3.get_items()


# Here are the 2 queries inputed during this run:

# Query #1
# input_user_init = """Current Market Situation:
# - Volatility Regime: low
# - Market Trend: bullish
# What rebalancing action should I take?"""

# Query #2
# I will not change stock exposure, however I'll decrease the bonds to 20% and increase the cash to 10%

#At the end of the discussion, the episodic memory will be built, using the current working session, and stored in the vector database.


User: Current Market Situation: - Volatility Regime: low - Market Trend: bullish  What rebalancing action should I take?
### Proposed Rebalancing Action

**Current Allocation:**
- **Stocks:** 70%
- **Bonds:** 25%
- **Cash:** 5%

**Market Situation:**
- **Volatility Regime:** Low
- **Market Trend:** Bullish

### Target Allocation
Given the current market conditions and insights from past episodes, consider the following target weights:
- **Stocks:** 75%
- **Bonds:** 20%
- **Cash:** 5%

### Proposed Trades
**1. Increase Equity Exposure:**
   - **Action:** Increase stock allocation from 70% to 75%.
   - **Percentage Change:** +5% towards stocks.
   - **Rationale:** With a low volatility environment and a bullish market trend, further increasing equity exposure aligns with the previous successful strategy of maximizing growth potential. The prior episodes showed that increased stock allocations during bullish trends frequently led to enhanced returns.

**2. Maintain Bond Allocation:**
   

In [ ]:
all_items

[{'content': "Current Market Situation: - Volatility Regime: low - Market Trend: bullish  What rebalancing action should I take?\n    Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:\n    \nConversation Summary: The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.\nPortfolio Allocation Trajectory: Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, B

In [ ]:
collection.get(ids=[collection_id])

{'ids': ['memory_2'],
 'embeddings': None,
 'documents': ["Conversation Summary: The market remains in a low volatility and bullish trend. The user has opted to maintain their stock allocation at 70%, while reducing their bond allocation to 20% and increasing cash reserves to 10%. This decision aims to leverage the current market conditions for potential growth while enhancing liquidity for future opportunities.\n  Portfolio Allocation Trajectory: Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market); Session 6: Stocks 70%, Bonds 20%, Cash 10% (Low volatility, bullish market)\n  What Worked: Maintaining a high equity exposure aligns well with the ongoing

In [ ]:
collection.get(
    ids=[collection_id],
    include=["embeddings"]
)


{'ids': ['memory_2'],
 'embeddings': array([[ 0.01819184,  0.01474414,  0.05563334, ..., -0.00739819,
          0.00739819, -0.00240457]]),
 'documents': None,
 'uris': None,
 'included': ['embeddings'],
 'data': None,
 'metadatas': None}

# Semantic Memory

Using the same logic (then in episodic memory) of appending the current context with new data, your can extract, using **RAG**, new information such as **facts** or **concepts** from a knowledge database.


This semantic memory is not about past experiences, but it's more about external knowledge your agent may need.

In porftolio rebalancing, this could mean adding **financial analysis** about **sectors** or **macro economic conditions**.

This kind of information can be stored in vector databases using RAG, allowing you to retrieve the most relevant insights when needed (related to the user query) and add them to the agent's context.

# Procedural Memory

A procedural memory is more "how to" perform a certain action.

For example, it could be, for a portfolio rebalancing:

* Identify the current portfolio allocation
* Compare it against the target allocation
* Calculate deviations for each asset class or sector
* Determine required buy/sell adjustments
* Check risk constraints (max exposure, volatility limits, liquidity)
* Propose rebalancing trades

